Analyse de Corrélation des Variables
Objectif : Déterminer s'il existe des relations mathématiques entre les différentes métriques de vente (ex: Est-ce que les produits les plus chers se vendent moins vite ?).

In [ ]:
import sys, os, pandas as pd, datetime as dt, matplotlib.pyplot as plt, seaborn as sns
sys.path.append(os.path.abspath(os.path.join('..')))
from src.analysis.connection import get_db_data 

# 1. Extraction
query = """
    SELECT customerNumber, MAX(orderDate) as last_order_date, COUNT(orderNumber) as frequency, 
    SUM(quantityOrdered * priceEach) as monetary 
    FROM orders JOIN orderdetails USING (orderNumber) GROUP BY customerNumber
"""
df_rfm = get_db_data(query)
df_rfm['last_order_date'] = pd.to_datetime(df_rfm['last_order_date'])

# 2. Calcul Récence
ref_date = df_rfm['last_order_date'].max() + dt.timedelta(days=1)
df_rfm['recency'] = (ref_date - df_rfm['last_order_date']).dt.days

# 3. Scoring
df_rfm['R'] = pd.qcut(df_rfm['recency'], 4, labels=[4, 3, 2, 1])
df_rfm['F'] = pd.qcut(df_rfm['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
df_rfm['M'] = pd.qcut(df_rfm['monetary'], 4, labels=[1, 2, 3, 4])

def assign_segment(row):
    total = int(row['R']) + int(row['F']) + int(row['M'])
    if total >= 10: return 'VIP'
    elif total >= 7: return 'Régulier'
    elif total >= 5: return 'À risque'
    else: return 'Perdu'

df_rfm['Segment'] = df_rfm.apply(assign_segment, axis=1)

# 4. Visualisation
plt.figure(figsize=(10, 6))
sns.countplot(data=df_rfm, x='Segment', palette='viridis', order=['VIP', 'Régulier', 'À risque', 'Perdu'])
plt.title('Analyse RFM : Segmentation Clients')
plt.savefig(os.path.join('..', 'visualisations', 'rfm_segmentation.png'), bbox_inches='tight')
plt.show()